<a href="https://colab.research.google.com/github/EJW1001001/tomato_disease_classification/blob/main/tomato_disease_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍅 Tomato Leaf Disease Classification 🍅
## Impact of Preprocessing Strategies on CNN Performance

---

### Project Overview

This notebook explores how different image preprocessing strategies affect a CNN's ability to classify tomato leaf diseases using the **PlantVillage** dataset. Four conditions are compared:

| Condition | Description |
|-----------|-------------|
| **Baseline** | No preprocessing — raw images |
| **Segmentation** | HSV-based leaf masking to isolate tissue |
| **Augmentation** | Random transforms to expand variation |
| **Seg + Aug** | Both strategies combined |

**Target classes:** Bacterial Spot · Yellow Leaf Curl Virus · Late Blight · Leaf Mold

> The central question: *Does isolating the leaf region help, or does it remove context the model actually needs?*

---
## 1. Installation & Imports

In [ ]:
!pip install tensorflow split-folders kagglehub -q

In [ ]:
import os
import shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import kagglehub

from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from IPython.display import display
import ipywidgets as widgets
import io

from tensorflow.keras import models, layers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

---
## 2. Global Configuration

All hyperparameters and paths are defined here. Modify this cell to control the full experiment.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_PATH        = "/kaggle/working/TomatoSubsetFiltered"
SEG_DATASET_PATH    = "/kaggle/working/TomatoSegmented"
SEG_AUG_DATASET_PATH= "/kaggle/working/TomatoSegAug"

# ── Dataset ──────────────────────────────────────────────────────────────────
INCLUDED_CLASSES = [
    "Tomato_Bacterial_spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
]
CLASS_DISPLAY_NAMES = [
    "Bacterial Spot",
    "Yellow Leaf Curl Virus",
    "Late Blight",
    "Leaf Mold",
]
MAX_IMAGES_PER_CLASS = 1000
NUM_CLASSES          = 4

# ── Image & Loader ────────────────────────────────────────────────────────────
IMG_SIZE    = (128, 128)
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2
SEED        = 42

# ── Segmentation ─────────────────────────────────────────────────────────────
HSV_LOWER_GREEN = np.array([25, 40, 40])
HSV_UPPER_GREEN = np.array([90, 255, 255])
MORPH_KERNEL    = np.ones((5, 5), np.uint8)

# ── Model Architecture ────────────────────────────────────────────────────────
INPUT_SHAPE    = (128, 128, 3)
DROPOUT_RATES  = (0.2, 0.3, 0.4, 0.4, 0.5)
L2_REG         = 0.001
DENSE_UNITS    = 256

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS         = 30
OPTIMIZER_TYPE = 'sgd'      # 'sgd' or 'adam'
LEARNING_RATE  = 0.01
MOMENTUM       = 0.9
PATIENCE_ES    = 10         # EarlyStopping patience
PATIENCE_LR    = 5          # ReduceLROnPlateau patience
MIN_LR         = 1e-6
LR_FACTOR      = 0.2

# ── Augmentation ─────────────────────────────────────────────────────────────
AUG_ROTATION    = 20
AUG_SHIFT       = 0.1
AUG_SHEAR       = 0.1
AUG_ZOOM        = 0.3
AUG_BRIGHTNESS  = [0.8, 1.2]

---
## 3. Data Loading & Preprocessing Functions

In [ ]:
def prepare_dataset():
    """Downloads PlantVillage and copies a filtered tomato subset."""
    print("Downloading PlantVillage dataset...")
    raw_path = os.path.join(kagglehub.dataset_download("emmarex/plantdisease"), "PlantVillage")

    if os.path.exists(DATASET_PATH):
        shutil.rmtree(DATASET_PATH)
    os.makedirs(DATASET_PATH)

    for cls in INCLUDED_CLASSES:
        src = os.path.join(raw_path, cls)
        dst = os.path.join(DATASET_PATH, cls)
        if not os.path.exists(src):
            print(f"  [!] Class not found: {cls}")
            continue
        os.makedirs(dst)
        imgs = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for f in tqdm(imgs[:MAX_IMAGES_PER_CLASS], desc=cls):
            shutil.copyfile(os.path.join(src, f), os.path.join(dst, f))

    print("\nClass distribution:")
    for cls in sorted(os.listdir(DATASET_PATH)):
        p = os.path.join(DATASET_PATH, cls)
        if os.path.isdir(p):
            print(f"  {cls}: {len(os.listdir(p))} images")


def segment_image(img_uint8):
    """Applies HSV green-channel masking to isolate leaf tissue."""
    hsv  = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, HSV_LOWER_GREEN, HSV_UPPER_GREEN)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, MORPH_KERNEL)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  MORPH_KERNEL)
    return cv2.bitwise_and(img_uint8, img_uint8, mask=mask)


def build_segmented_dataset(input_dir, output_dir):
    """Walks input_dir, segments each image, and saves to output_dir."""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    for subdir, _, files in os.walk(input_dir):
        for file in files:
            if not file.lower().endswith(('.jpg','.jpeg','.png')):
                continue
            in_path  = os.path.join(subdir, file)
            out_path = os.path.join(output_dir, os.path.relpath(in_path, input_dir))
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            try:
                img      = Image.open(in_path).convert('RGB').resize(IMG_SIZE)
                img_u8   = np.array(img)
                segmented = segment_image(img_u8)
                Image.fromarray(segmented).save(out_path)
            except Exception as e:
                print(f"  [!] Error: {in_path} — {e}")


def make_baseline_generators():
    """Plain rescaled generators — no augmentation, no segmentation."""
    gen = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)
    return _flow(gen, DATASET_PATH)


def make_segmented_generators():
    """Generators over the HSV-segmented dataset."""
    build_segmented_dataset(DATASET_PATH, SEG_DATASET_PATH)
    gen = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)
    return _flow(gen, SEG_DATASET_PATH)


def make_augmented_generators():
    """Generators with random augmentation over the original dataset."""
    gen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=AUG_ROTATION,
        width_shift_range=AUG_SHIFT,
        height_shift_range=AUG_SHIFT,
        shear_range=AUG_SHEAR,
        zoom_range=AUG_ZOOM,
        horizontal_flip=True,
        brightness_range=AUG_BRIGHTNESS,
        fill_mode='nearest',
        validation_split=VAL_SPLIT
    )
    return _flow(gen, DATASET_PATH)


def make_seg_aug_generators():
    """Generators combining segmentation and augmentation."""
    build_segmented_dataset(DATASET_PATH, SEG_AUG_DATASET_PATH)
    gen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=AUG_ROTATION,
        width_shift_range=AUG_SHIFT,
        height_shift_range=AUG_SHIFT,
        shear_range=AUG_SHEAR,
        zoom_range=AUG_ZOOM,
        horizontal_flip=True,
        brightness_range=AUG_BRIGHTNESS,
        fill_mode='nearest',
        validation_split=VAL_SPLIT
    )
    return _flow(gen, SEG_AUG_DATASET_PATH)


def _flow(datagen, path):
    """Helper: creates train and val generators from a datagen and path."""
    train = datagen.flow_from_directory(
        path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='training', seed=SEED
    )
    val = datagen.flow_from_directory(
        path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='validation', seed=SEED
    )
    return train, val

---
## 4. Visual Samples — Preprocessing Comparison

Below we visualize how each preprocessing strategy transforms a raw leaf image before it reaches the model.

In [ ]:
def _random_image_path(dataset_dir):
    """Returns a random image path from dataset_dir."""
    for cls_dir in os.listdir(dataset_dir):
        cls_path = os.path.join(dataset_dir, cls_dir)
        if os.path.isdir(cls_path):
            imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if imgs:
                return os.path.join(cls_path, imgs[0])


def plot_preprocessing_samples():
    """Shows one image under all four preprocessing conditions."""
    img_path = _random_image_path(DATASET_PATH)
    img      = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
    img_u8   = np.array(img)

    # Augmentation preview
    aug_gen = ImageDataGenerator(
        rotation_range=AUG_ROTATION, width_shift_range=AUG_SHIFT,
        height_shift_range=AUG_SHIFT, shear_range=AUG_SHEAR,
        zoom_range=AUG_ZOOM, horizontal_flip=True,
        brightness_range=AUG_BRIGHTNESS, fill_mode='nearest'
    )
    aug_iter  = aug_gen.flow(np.expand_dims(img_u8, 0), batch_size=1)
    aug_img   = next(aug_iter)[0].astype(np.uint8)
    seg_img   = segment_image(img_u8)
    seg_aug   = segment_image(aug_img)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle("Preprocessing Strategy Comparison", fontsize=14, fontweight='bold', y=1.02)

    samples = [
        (img_u8,  "Baseline",        "No preprocessing"),
        (seg_img, "Segmentation",    "HSV green-channel mask"),
        (aug_img, "Augmentation",    "Random transforms"),
        (seg_aug, "Seg + Aug",       "Mask + transforms"),
    ]
    for ax, (arr, title, subtitle) in zip(axes, samples):
        ax.imshow(arr)
        ax.set_title(f"{title}\n{subtitle}", fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


def plot_augmentation_gallery(n=6):
    """Shows n augmented versions of a single image."""
    img_path = _random_image_path(DATASET_PATH)
    img_u8   = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))

    aug_gen = ImageDataGenerator(
        rotation_range=AUG_ROTATION, width_shift_range=AUG_SHIFT,
        height_shift_range=AUG_SHIFT, shear_range=AUG_SHEAR,
        zoom_range=AUG_ZOOM, horizontal_flip=True,
        brightness_range=AUG_BRIGHTNESS, fill_mode='nearest'
    )
    aug_iter = aug_gen.flow(np.expand_dims(img_u8, 0), batch_size=1)

    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    fig.suptitle("Augmentation Gallery — Same Image, Different Transforms",
                 fontsize=12, fontweight='bold')
    for ax in axes:
        ax.imshow(next(aug_iter)[0].astype(np.uint8))
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def plot_segmentation_gallery(n=4):
    """Shows segmented samples from each class."""
    class_dirs = [d for d in sorted(os.listdir(DATASET_PATH))
                  if os.path.isdir(os.path.join(DATASET_PATH, d))]

    fig, axes = plt.subplots(2, len(class_dirs), figsize=(4*len(class_dirs), 8))
    fig.suptitle("Segmentation per Class — Original vs Masked",
                 fontsize=13, fontweight='bold')

    for i, cls in enumerate(class_dirs):
        cls_path = os.path.join(DATASET_PATH, cls)
        imgs     = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        img_u8   = np.array(Image.open(os.path.join(cls_path, imgs[0])).convert('RGB').resize(IMG_SIZE))
        seg      = segment_image(img_u8)
        label    = cls.replace('Tomato_', '').replace('Tomato__', '').replace('_', ' ')

        axes[0, i].imshow(img_u8);  axes[0, i].set_title(label, fontsize=9); axes[0, i].axis('off')
        axes[1, i].imshow(seg);     axes[1, i].set_title('Segmented', fontsize=9); axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()

---
## 5. Model Definition & Training

In [ ]:
def build_model():
    """Builds the custom CNN architecture."""
    reg = tf.keras.regularizers.l2(L2_REG)
    m = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(32,  (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(DROPOUT_RATES[0]),

        layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(DROPOUT_RATES[1]),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(DROPOUT_RATES[2]),

        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(DROPOUT_RATES[3]),

        layers.Flatten(),
        layers.Dense(DENSE_UNITS, activation='relu', kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Dropout(DROPOUT_RATES[4]),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return m


def train_model(train_gen, val_gen, save_path='best_model.keras'):
    """Compiles and trains the CNN; returns (model, history)."""
    model = build_model()

    if OPTIMIZER_TYPE == 'sgd':
        opt = tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE, momentum=MOMENTUM)
    else:
        opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    cbs = [
        callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=LR_FACTOR, patience=PATIENCE_LR, min_lr=MIN_LR),
        callbacks.ModelCheckpoint(save_path, monitor='val_accuracy', save_best_only=True),
    ]

    history = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS, callbacks=cbs, verbose=1
    )
    return model, history

---
## 6. Evaluation & Visualization Functions

In [ ]:
def get_predictions(model, val_gen):
    """Runs inference on val_gen and returns (true_labels, pred_labels)."""
    val_gen.reset()
    pred_probs, true_labels = [], []
    for i, (imgs, lbls) in enumerate(val_gen):
        pred_probs.extend(model.predict(imgs, verbose=0))
        true_labels.extend(np.argmax(lbls, axis=1))
        if (i + 1) == len(val_gen):
            break
    return np.array(true_labels), np.argmax(pred_probs, axis=1)


def evaluate_model(model, history, val_gen, label="Model"):
    """Prints metrics, confusion matrix, and training curves for one model."""
    true_labels, pred_labels = get_predictions(model, val_gen)
    class_names = list(val_gen.class_indices.keys())

    # Confusion matrix
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix — {label}", fontweight='bold')
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()

    # Metrics
    print(f"\n{'─'*50}")
    print(f" {label} — Validation Metrics")
    print(f"{'─'*50}")
    print(f"  Accuracy : {accuracy_score(true_labels, pred_labels):.4f}")
    print(f"  Precision: {precision_score(true_labels, pred_labels, average='weighted'):.4f}")
    print(f"  Recall   : {recall_score(true_labels, pred_labels, average='weighted'):.4f}")
    print(f"  F1 Score : {f1_score(true_labels, pred_labels, average='weighted'):.4f}")
    print(f"\n{classification_report(true_labels, pred_labels, target_names=class_names)}")

    # Training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Training Curves — {label}", fontweight='bold')

    ax1.plot(history.history['accuracy'],     label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)

    ax2.plot(history.history['loss'],     label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)

    plt.tight_layout(); plt.show()


def plot_all_histories(histories, labels):
    """Overlays training and validation curves for all models."""
    for metric in ('accuracy', 'loss'):
        fig, (ax_tr, ax_val) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"{metric.capitalize()} Comparison — All Conditions", fontweight='bold')

        for h, lbl in zip(histories, labels):
            ax_tr.plot(h.history[metric],           label=lbl)
            ax_val.plot(h.history[f'val_{metric}'], label=lbl)

        for ax, title in zip((ax_tr, ax_val), ('Training', 'Validation')):
            ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
            ax.legend(); ax.grid(True)

        plt.tight_layout(); plt.show()


def plot_metric_summary(results):
    """Bar chart comparing final metrics across all four conditions."""
    labels  = list(results.keys())
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
    x = np.arange(len(labels))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 5))
    for i, m in enumerate(metrics):
        vals = [results[l][m] for l in labels]
        ax.bar(x + i*width, vals, width, label=m)

    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Final Metric Comparison — All Preprocessing Conditions', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.4)
    plt.tight_layout(); plt.show()


def compute_metrics(model, val_gen):
    """Returns a dict of scalar metrics for a model."""
    true, pred = get_predictions(model, val_gen)
    return {
        'Accuracy' : accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall'   : recall_score(true, pred, average='weighted'),
        'F1'       : f1_score(true, pred, average='weighted'),
    }

---
## 7. Inference Widget

In [ ]:
def launch_predictor(model):
    """Upload-based leaf disease predictor widget for Colab."""
    upload = widgets.FileUpload(accept='image/*', multiple=False)
    output = widgets.Output()

    def _on_upload(change):
        output.clear_output()
        for _, info in upload.value.items():
            img = Image.open(io.BytesIO(info['content'])).convert('RGB').resize(IMG_SIZE)
            arr = np.expand_dims(np.array(img) / 255.0, axis=0)
            probs = model.predict(arr, verbose=0)[0]
            idx   = np.argmax(probs)
            with output:
                print(f"Prediction : {CLASS_DISPLAY_NAMES[idx]}")
                print(f"Confidence : {probs[idx]*100:.2f}%")
                plt.imshow(img); plt.axis('off')
                plt.title(f"{CLASS_DISPLAY_NAMES[idx]} ({probs[idx]*100:.1f}%)")
                plt.show()

    upload.observe(_on_upload, names='value')
    display(widgets.HTML("<h3>🍅 Tomato Leaf Disease Classifier</h3>"))
    display(upload, output)

---
## 8. GPU Check

In [ ]:
gpu = tf.test.gpu_device_name()
print(f"GPU: {gpu}" if gpu == '/device:GPU:0' else "⚠️  No GPU detected. Set Runtime → GPU for faster training.")

---
## 9. Main Pipeline

All training runs are executed sequentially below. Each model is saved, evaluated, and compared.

In [ ]:
# ── 1. Prepare dataset ───────────────────────────────────────────────────────
prepare_dataset()

In [ ]:
# ── 2. Visual samples ────────────────────────────────────────────────────────
plot_preprocessing_samples()
plot_segmentation_gallery()
plot_augmentation_gallery()

In [ ]:
# ── 3. Train — Baseline ──────────────────────────────────────────────────────
train_base, val_base         = make_baseline_generators()
model_base, hist_base        = train_model(train_base, val_base, 'model_baseline.keras')
evaluate_model(model_base, hist_base, val_base, label="Baseline")

In [ ]:
# ── 4. Train — Segmentation ──────────────────────────────────────────────────
train_seg, val_seg           = make_segmented_generators()
model_seg, hist_seg          = train_model(train_seg, val_seg, 'model_seg.keras')
evaluate_model(model_seg, hist_seg, val_seg, label="Segmentation")

In [ ]:
# ── 5. Train — Augmentation ──────────────────────────────────────────────────
train_aug, val_aug           = make_augmented_generators()
model_aug, hist_aug          = train_model(train_aug, val_aug, 'model_aug.keras')
evaluate_model(model_aug, hist_aug, val_aug, label="Augmentation")

In [ ]:
# ── 6. Train — Segmentation + Augmentation ───────────────────────────────────
train_sa, val_sa             = make_seg_aug_generators()
model_sa, hist_sa            = train_model(train_sa, val_sa, 'model_seg_aug.keras')
evaluate_model(model_sa, hist_sa, val_sa, label="Seg + Aug")

In [ ]:
# ── 7. Cross-model comparison ────────────────────────────────────────────────
CONDITION_LABELS = ["Baseline", "Segmentation", "Augmentation", "Seg + Aug"]

plot_all_histories(
    [hist_base, hist_seg, hist_aug, hist_sa],
    CONDITION_LABELS
)

results = {
    "Baseline"    : compute_metrics(model_base, val_base),
    "Segmentation": compute_metrics(model_seg,  val_seg),
    "Augmentation": compute_metrics(model_aug,  val_aug),
    "Seg + Aug"   : compute_metrics(model_sa,   val_sa),
}

plot_metric_summary(results)

print("\n Final Results Summary")
print(f"{'Condition':<18} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("─" * 60)
for cond, m in results.items():
    print(f"{cond:<18} {m['Accuracy']:>10.4f} {m['Precision']:>10.4f} {m['Recall']:>10.4f} {m['F1']:>10.4f}")

In [ ]:
# ── 8. Inference widget (best model) ─────────────────────────────────────────
launch_predictor(model_aug)

## 10. Conclusions

### Summary of Findings

The experiment evaluated four preprocessing strategies on a four-class tomato leaf disease dataset using a consistent CNN architecture.

---

#### 1. Augmentation — Best Overall Performer

Data augmentation consistently achieved the highest or near-highest validation metrics. By exposing the model to rotations, shifts, brightness changes, and zoom variations, it learns more robust and generalisable features — directly addressing the limited dataset size. For this type of classification task, where appearance can vary considerably under natural conditions, augmentation provides tangible benefits without discarding any visual information.

---

#### 2. Segmentation — Modest Improvement at a Cost

HSV-based leaf segmentation showed marginal improvement over the baseline in some runs, but the gains were inconsistent. The core issue with this dataset is that the leaf backgrounds are already relatively uniform — green tissue dominates the frame. In contrast, the disease patterns (spots, yellowing, mold) are the most discriminative visual features, and these are precisely where the segmentation mask can introduce noise or loss of information if the colour threshold is imperfect. The improvement seen is real but modest.

---

#### 3. Segmentation + Augmentation — Worst Combined Performance

Counterintuitively, combining both strategies produced the worst results. This appears to be an interaction effect: augmentation (especially brightness variation) disrupts the fixed HSV thresholds used for segmentation, producing masks that are inconsistently applied across augmented variants. The model therefore receives inputs with erratic masked regions, increasing noise without adding useful variation. The two techniques, individually reasonable, interfere when composed in this order.

---

### Practical Recommendation

> For datasets where the subject already dominates the frame and disease patterns are the primary discriminative signal, **augmentation alone is the most effective and reliable preprocessing strategy**. Segmentation should only be applied when background clutter is a genuine problem — and never after augmentation, which degrades mask quality.

---
